# Day 60: ARIMA Forecasting

## Install Note
ARIMA is one of the most widely used classical statistical models for forecasting a single time series based purely on its own past values (as opposed to ML models that use external features). It lives in `statsmodels` rather than `scikit-learn` because it's a statistical time series model with its own fitting procedure (maximum likelihood estimation) rather than a general-purpose supervised learning algorithm. If `statsmodels` isn't already installed, run the pip command below once in your environment.

In [ ]:
# pip install statsmodels

## Preparing the Series
ARIMA expects a `pandas.Series` indexed by time (ideally with a regular frequency like monthly or daily) since it models how each value relates to previous values and past forecasting errors in sequence — shuffled or irregularly-spaced data breaks that assumption entirely. It also works best when you already have a rough sense of the series' trend/seasonality (e.g. from decomposition, like Day 59), since that understanding guides how you choose ARIMA's `(p, d, q)` order below. Here we simulate a random walk with drift: starting at 100 and cumulatively summing random monthly increments (average +1 per step, with noise) using `np.cumsum`, which produces a series that trends upward but wanders unpredictably — a reasonable stand-in for something like a slowly growing but noisy business metric.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(1)
dates = pd.date_range('2022-01-01', periods=60, freq='M')
# Random walk with drift: cumulatively sum random monthly changes (mean +1, std 3)
# starting from a base of 100 -- produces a noisy but generally upward-trending series
series = pd.Series(100 + np.cumsum(np.random.normal(1, 3, 60)), index=dates)

## Fitting an ARIMA Model
ARIMA stands for AutoRegressive Integrated Moving Average, and its `(p, d, q)` order controls three separate pieces: `p` is the number of autoregressive (AR) terms — how many past values of the series are used to predict the next one; `d` is the degree of differencing (Integrated) — how many times you subtract each value from the previous one to make a trending series stationary (roughly, flat/stable in mean and variance); and `q` is the number of moving-average (MA) terms — how many past forecast errors are used to correct the next prediction. Here `order=(1, 1, 1)` means: use 1 lagged value, difference the series once to remove the trend, and use 1 lagged forecast error. The `.summary()` output includes fitted coefficients for each AR/MA term along with p-values (testing whether each term is statistically distinguishable from zero) and information criteria (like AIC) useful for comparing different orders against each other.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# order=(p, d, q): p=1 autoregressive term, d=1 differencing to remove the trend, q=1 moving-average term
model = ARIMA(series, order=(1, 1, 1))
fitted = model.fit()
print(fitted.summary())

## Forecasting Future Values
Once fitted, the model can project forward beyond the observed data. `get_forecast(steps=6)` produces predictions for the next 6 time periods (6 months, matching our data's frequency) past the end of the training series. Unlike a plain point forecast, `get_forecast` also gives you a confidence interval (`conf_int()`) — a range within which the true future value is expected to fall with a given probability (95% by default) — which matters because a single predicted number hides how uncertain that prediction actually is; the interval should widen the further out you forecast, reflecting growing uncertainty about the more distant future.

In [ ]:
# Forecast the next 6 periods beyond the observed series
forecast = fitted.get_forecast(steps=6)
print(forecast.predicted_mean)  # point forecasts
print(forecast.conf_int())  # 95% confidence interval (lower, upper) for each forecasted point